# Job Search Agent (LangGraph + RAG + Tool Calling)

Monitors a configurable list of companies' careers pages for a configurable job title.

- **Orchestration**: LangGraph's `create_react_agent` (a minimal ReAct loop)
- **Tool calling**: the LLM decides which company to check and calls a search tool
- **RAG**: each careers page is fetched, chunked, embedded (open-source HF sentence-transformer), and searched with FAISS
- **LLM**: open-source HuggingFace chat model, loaded locally into the runtime (4-bit quantized on GPU) — no HF API key needed
- **Tracing (optional)**: Arize Phoenix — open source, runs locally, no API key needed

Run cells top to bottom. Re-run the last cell any time you want to re-check (that's your "monitor").

In [ ]:
!pip -q install langgraph langchain langchain-huggingface langchain-community faiss-cpu sentence-transformers beautifulsoup4 requests arize-phoenix openinference-instrumentation-langchain transformers accelerate bitsandbytes

In [ ]:
import os
import requests
from bs4 import BeautifulSoup

from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.prebuilt import create_react_agent

# Optional: local tracing with Arize Phoenix (open source, no API key needed)
ENABLE_TRACING = True
if ENABLE_TRACING:
    import phoenix as px
    from phoenix.otel import register
    from openinference.instrumentation.langchain import LangChainInstrumentor

    px.launch_app()  # opens the Phoenix UI at http://localhost:6006
    tracer_provider = register(project_name="job-search-agent")
    LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

## Configure: companies + job to watch for

Replace the placeholder URLs below with the real careers page URLs you want to track. Static/server-rendered pages (e.g. Greenhouse/Lever job boards) work best with the simple `requests` fetch used here — heavily JS-rendered pages may return little text.

In [ ]:
COMPANIES = {
    "OpenAI": "https://jobs.ashbyhq.com/openai/",
    "Perplexity": "https://jobs.ashbyhq.com/perplexity/",
    "Baseten": "https://jobs.ashbyhq.com/baseten/",
    "Fireworks": "https://jobs.ashbyhq.com/fireworks/",
    "Anthropic": "https://job-boards.greenhouse.io/anthropic",
}

TARGET_JOB = "Forward Deployed Engineer"  # job title / keywords to watch for

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # any HF chat model that supports tool calling

## Tools: fetch + RAG search over each careers page

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

def _page_text(url: str) -> str:
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15).text
    return BeautifulSoup(html, "html.parser").get_text(separator="\n", strip=True)

@tool
def list_companies() -> str:
    """List the company names currently being tracked."""
    return "\n".join(COMPANIES.keys())

@tool
def search_careers_page(company: str, query: str) -> str:
    """RAG search over one company's careers page. Fetches the page, splits it
    into chunks, embeds them, and returns the chunks most relevant to `query`
    (e.g. a job title). `company` must be one of the tracked company names."""
    url = COMPANIES.get(company)
    if not url:
        return f"Unknown company '{company}'. Known companies: {list(COMPANIES)}"
    text = _page_text(url)
    docs = splitter.create_documents([text])
    if not docs:
        return "Page had no readable text."
    store = FAISS.from_documents(docs, embeddings)
    hits = store.similarity_search(query, k=3)
    return "\n---\n".join(d.page_content for d in hits) or "No relevant content found."

tools = [list_companies, search_careers_page]

## Agent: a minimal LangGraph ReAct loop

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

# Loads MODEL_ID's weights into the GPU runtime (4-bit quantized) instead of
# calling the HF Inference API, so no HUGGINGFACEHUB_API_TOKEN is needed.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
)

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    return_full_text=False,
)

llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=hf_pipeline))

SYSTEM_PROMPT = (
    "You monitor company careers pages for a specific job opening. "
    "Use `list_companies` to see which companies to check, then call "
    "`search_careers_page` once per company with the target job as the query. "
    "Finish with a short report: for each company say FOUND or NOT FOUND, "
    "and quote the matching job title/snippet when FOUND."
)

agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

## Run the check (re-run this cell any time to monitor again)

In [ ]:
result = agent.invoke({
    "messages": [("user", f"Check every tracked company for openings matching: {TARGET_JOB}")]
})

print(result["messages"][-1].content)

If `ENABLE_TRACING` is left on, the full trace (every tool call + LLM step) is visible in the Phoenix UI, opened automatically at http://localhost:6006 under the `job-search-agent` project. Phoenix is open source and runs entirely locally — no account or API key needed.